In [ ]:
!pip install convokit transformers datasets accelerate scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/cga_data"
SAVE_PATH = "/content/drive/MyDrive/cga_deberta_final"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
from convokit import Corpus, download
from sklearn.model_selection import train_test_split

corpus = Corpus(filename=download("conversations-gone-awry-corpus"))
corpus.print_summary_stats()

# extract turns with labels
rows = []
for utt in corpus.iter_utterances():
    if utt.meta.get("is_section_header", False):
        continue
    text = utt.text
    if not text or len(text.strip()) < 5:
        continue
    label = int(utt.meta.get("comment_has_personal_attack", 0))
    rows.append({"text": text, "label": label})

df = pd.DataFrame(rows)
print(f"Total turns: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# balance: undersample negatives 3:1
df_pos = df[df["label"] == 1]
df_neg = df[df["label"] == 0].sample(n=len(df_pos) * 3, random_state=42)
df_balanced = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

# split
train_df, test_df = train_test_split(df_balanced, test_size=0.15, stratify=df_balanced["label"], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=42)

# save to Drive
train_df.to_csv(f"{DATA_DIR}/train.csv", index=False)
val_df.to_csv(f"{DATA_DIR}/val.csv", index=False)
test_df.to_csv(f"{DATA_DIR}/test.csv", index=False)
print(f"Saved to {DATA_DIR}/ -- Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem
Number of Speakers: 8069
Number of Utterances: 30021
Number of Conversations: 4188
Total turns: 25767
Label distribution:
label
0    23673
1     2094
Name: count, dtype: int64
Saved to /content/drive/MyDrive/cga_data/ -- Train: 6407 | Val: 712 | Test: 1257


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Loaded -- Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Loaded -- Train: 6407 | Val: 712 | Test: 1257


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/6407 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/1257 [00:00<?, ? examples/s]

In [ ]:
# ============================================================
# CELL 5 (FIXED): Switch to roberta-base
# ============================================================
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

# re-tokenize with roberta tokenizer
train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = np.argmax(logits, axis=-1)
    auc = roc_auc_score(labels, probs)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "f1": report["1"]["f1-score"],
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "auc": auc,
    }

training_args = TrainingArguments(
    output_dir="./cga_roberta",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    bf16=True,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1257 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Parameters: 124.6M


In [ ]:
# ============================================================
# DEBUG: Check what's actually in your data
# ============================================================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

print("=== Label counts (full train) ===")
print(train_df["label"].value_counts())
print(f"\n=== Label dtype: {train_df['label'].dtype} ===")

# Check some positive examples
pos = train_df[train_df["label"] == 1]
print(f"\n=== Positive samples: {len(pos)} ===")
if len(pos) > 0:
    for t in pos["text"].head(5):
        print(f"  [+] {t[:120]}")

# Check some negative examples
neg = train_df[train_df["label"] == 0]
print(f"\n=== Negative samples: {len(neg)} ===")
for t in neg["text"].head(3):
    print(f"  [-] {t[:120]}")

# Check val set size (suspiciously looks like only 4 samples)
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
print(f"\n=== Val set: {len(val_df)} samples ===")
print(val_df["label"].value_counts())

=== Label counts (full train) ===
label
0    4805
1    1602
Name: count, dtype: int64

=== Label dtype: int64 ===

=== Positive samples: 1602 ===
  [+] Thats bullshit, its complete opinion presented as fact.  Who decided that he "destroyed credibility" who are you to judg
  [+] Excuse me, but the Better Business Bureau site isnt a reliable source? What do you need, a picture book?  The subject is
  [+] WOW, that's some major hype. Why are you wasting people's time with such trivial things? No where does that ad say somet
  [+] I asked you for evidence. Provide the quote to Gibb or you are simply a liar inventing a source to defend your vendetta.
  [+] RUBBISH! Libertarianism is a modern ideology - it's not anything or anybody u label it as. -

=== Negative samples: 4805 ===
  [-] Well considering that 'folk metal' encompases rather a diverse number of different styles of music , and given that genr
  [-] "Epic" is not necessarily a value judgment, but can be a descriptive word. It is a

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
1,0.115179,0.107177,0.964888,0.927954,0.952663,0.904494,0.991478
2,0.068390,0.124578,0.976124,0.952909,0.939891,0.966292,0.992415
3,0.040229,0.130173,0.980337,0.960674,0.960674,0.960674,0.994629
4,0.014246,0.126543,0.978933,0.957983,0.955307,0.960674,0.993988
5,0.001466,0.150898,0.974719,0.950000,0.939560,0.960674,0.994140


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1005, training_loss=0.09807413392723423, metrics={'train_runtime': 73.1031, 'train_samples_per_second': 438.217, 'train_steps_per_second': 13.748, 'total_flos': 4214381329228800.0, 'train_loss': 0.09807413392723423, 'epoch': 5.0})

In [ ]:
results = trainer.evaluate(test_ds)
print("\n=== Test Results ===")
for k, v in results.items():
    if k.startswith("eval_"):
        print(f"  {k.replace('eval_', '')}: {v:.4f}")


=== Test Results ===
  loss: 0.1206
  accuracy: 0.9761
  f1: 0.9522
  precision: 0.9522
  recall: 0.9522
  auc: 0.9962
  runtime: 0.6242
  samples_per_second: 2013.7790
  steps_per_second: 32.0410


In [ ]:
# ============================================================
# CELL 8 (FIXED): Save with corrected LayerNorm keys
# ============================================================
import copy

state_dict = copy.deepcopy(model.state_dict())
fixed_state_dict = {}
for k, v in state_dict.items():
    new_k = k.replace(".beta", ".bias").replace(".gamma", ".weight")
    fixed_state_dict[new_k] = v

model.load_state_dict(fixed_state_dict, strict=False)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved (fixed keys) to {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved (fixed keys) to /content/drive/MyDrive/cga_deberta_final


In [ ]:
# ============================================================
# CELL 9 (FIXED): Inference using trainer's model directly
# ============================================================

# First, test with the in-memory model (should work)
model.eval()
device = model.device

def get_toxicity_score(text: str) -> float:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    return probs[0, 1].item()

samples = [
    "I think we should consider both perspectives here.",
    "You clearly have no idea what you're talking about, just stop.",
    "That's an interesting point, but the data suggests otherwise.",
    "You're a complete idiot and a liar.",
    "RUBBISH! You don't know what you're talking about.",
]
for s in samples:
    print(f"  [{get_toxicity_score(s):.3f}] {s[:80]}")

  [0.000] I think we should consider both perspectives here.
  [0.001] You clearly have no idea what you're talking about, just stop.
  [0.000] That's an interesting point, but the data suggests otherwise.
  [0.999] You're a complete idiot and a liar.
  [0.999] RUBBISH! You don't know what you're talking about.


In [ ]:
# ============================================================
# CELL: Compare fine-tuned vs base RoBERTa on subtle toxicity
# ============================================================

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# --- Load base (unfine-tuned) RoBERTa ---
base_model_name = "roberta-base"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2).to("cuda").eval()

# --- Fine-tuned model already in memory as `model` ---
model.eval()

def score_with_model(text, m, tok):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    with torch.no_grad():
        logits = m(**inputs).logits
    return torch.nn.functional.softmax(logits, dim=-1)[0, 1].item()

# --- Test samples: explicit → subtle → civil ---
samples = [
    # Explicit attacks
    ("EXPLICIT", "You're a complete idiot and a liar."),
    ("EXPLICIT", "Shut the hell up, nobody asked you."),
    # Subtle / debate toxicity
    ("SUBTLE", "You clearly don't understand how this works."),
    ("SUBTLE", "That's a strawman and you know it."),
    ("SUBTLE", "Let me explain this in simpler terms for you."),
    ("SUBTLE", "I'm not sure you've actually read the source material."),
    ("SUBTLE", "That's a remarkably naive take."),
    ("SUBTLE", "Sure, keep telling yourself that."),
    ("SUBTLE", "Anyone with basic reading comprehension would disagree."),
    ("SUBTLE", "I see you're just parroting talking points now."),
    # Civil
    ("CIVIL", "I think we should consider both perspectives here."),
    ("CIVIL", "That's an interesting point, but the data suggests otherwise."),
    ("CIVIL", "I respectfully disagree with that interpretation."),
    ("CIVIL", "Could you provide a source for that claim?"),
]

print(f"{'TYPE':<10} {'FINE-TUNED':>10} {'BASE':>10}  TEXT")
print("-" * 90)
for label, text in samples:
    ft_score = score_with_model(text, model, tokenizer)
    base_score = score_with_model(text, base_model, base_tokenizer)
    print(f"{label:<10} {ft_score:>10.3f} {base_score:>10.3f}  {text[:60]}")

# cleanup base model from GPU
del base_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TYPE       FINE-TUNED       BASE  TEXT
------------------------------------------------------------------------------------------
EXPLICIT        0.999      0.505  You're a complete idiot and a liar.
EXPLICIT        1.000      0.507  Shut the hell up, nobody asked you.
SUBTLE          0.001      0.504  You clearly don't understand how this works.
SUBTLE          0.002      0.507  That's a strawman and you know it.
SUBTLE          0.001      0.503  Let me explain this in simpler terms for you.
SUBTLE          0.000      0.502  I'm not sure you've actually read the source material.
SUBTLE          0.001      0.504  That's a remarkably naive take.
SUBTLE          0.001      0.507  Sure, keep telling yourself that.
SUBTLE          0.001      0.504  Anyone with basic reading comprehension would disagree.
SUBTLE          0.000      0.508  I see you're just parroting talking points now.
CIVIL           0.000      0.502  I think we should consider both perspectives here.
CIVIL           0.000 